# Week 3 · Solutions

In [ ]:
import numpy as np
from pathlib import Path
DATA = next(p / "data" / "generated" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "make_data.py").exists())
np.set_printoptions(precision=3, suppress=True)
print("NumPy", np.__version__)

### Exercise 1 · Build arrays

In [ ]:
def frame():
    f = np.ones((5, 5), dtype=int)
    f[1:-1, 1:-1] = 0
    return f

In [ ]:
f = frame()
assert f.shape == (5, 5) and f.dtype.kind == 'i'
assert f.sum() == 16 and f[2, 2] == 0 and f[0].sum() == 5
print("✓ exercise 1")

### Exercise 2 · Views vs copies

In [ ]:
def double_evens(a):
    r = a.copy()
    r[::2] *= 2
    return r

In [ ]:
a = np.arange(6)
r = double_evens(a)
assert list(r) == [0, 1, 4, 3, 8, 5] and list(a) == [0, 1, 2, 3, 4, 5]
print("✓ exercise 2")

### Exercise 3 · Masking

In [ ]:
def clip_gpa(g):
    out = g.copy()
    changed = (out < 0) | (out > 4)
    out[out < 0] = 0
    out[out > 4] = 4
    return out, int(changed.sum())

In [ ]:
arr, n = clip_gpa(np.array([-1.0, 2.5, 4.7, 4.0]))
assert list(arr) == [0.0, 2.5, 4.0, 4.0] and n == 2
print("✓ exercise 3")

### Exercise 4 · Broadcasting

In [ ]:
def normalise_rows(M):
    s = M.sum(axis=1, keepdims=True)
    return np.divide(M, s, out=np.zeros_like(M, dtype=float), where=s != 0)

In [ ]:
M = np.array([[1., 1., 2.], [0., 0., 0.], [5., 0., 0.]])
R = normalise_rows(M)
assert np.allclose(R, [[0.25, 0.25, 0.5], [0, 0, 0], [1, 0, 0]]), R
assert not np.isnan(R).any()
print("✓ exercise 4")

### Exercise 5 · Axis aggregation

In [ ]:
def hardest_exam_and_best_student(S):
    return int(S.std(axis=0).argmax()), int(S.mean(axis=1).argmax())

In [ ]:
S = np.array([[80, 90, 70], [60, 85, 95], [75, 65, 80], [90, 95, 60]])
assert hardest_exam_and_best_student(S) == (2, 3)
print("✓ exercise 5")

### Exercise 6 · Reproducible sampling

In [ ]:
def sample_means(seed, n_samples, size):
    rng = np.random.default_rng(seed)
    return rng.normal(0, 1, size=(n_samples, size)).mean(axis=1)

In [ ]:
m = sample_means(7, 1000, 25)
assert m.shape == (1000,)
assert np.array_equal(m, sample_means(7, 1000, 25))        # reproducible
assert abs(m.mean()) < 0.05 and abs(m.std() - 0.2) < 0.03  # CLT: sd ≈ 1/√25
print("✓ exercise 6")

### Exercise 7 · Pairwise distances

In [ ]:
def pairwise(P):
    diff = P[:, None, :] - P[None, :, :]        # (n, 1, 2) - (1, n, 2) → (n, n, 2)
    return np.sqrt((diff ** 2).sum(axis=2))

In [ ]:
P = np.array([[0, 0], [3, 4], [6, 8]])
D = pairwise(P)
assert D.shape == (3, 3) and np.allclose(D, [[0, 5, 10], [5, 0, 5], [10, 5, 0]])
print("✓ exercise 7")

### Exercise 8 · Linear regression by hand

In [ ]:
def fit_line(x, y):
    X = np.column_stack([x, np.ones_like(x)])
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    return float(beta[0]), float(beta[1])

In [ ]:
x = np.arange(20, dtype=float)
y = 3 * x - 2
m, c = fit_line(x, y)
assert abs(m - 3) < 1e-9 and abs(c + 2) < 1e-9
print("✓ exercise 8")

### Exercise 9 · Real data

In [ ]:
def revenue_by_city(path):
    d = np.genfromtxt(path, delimiter=",", names=True, dtype=None, encoding="utf-8")
    rev = d["units"] * d["unit_price"]
    return {str(c): round(float(rev[d["city"] == c].mean()), 2) for c in np.unique(d["city"])}

In [ ]:
r = revenue_by_city(DATA / "sales_2025.csv")
assert set(r) == {"Amman", "Irbid", "Zarqa", "Aqaba", "Salt", "Madaba"}
assert all(1000 < v < 10000 for v in r.values()), r
print("✓ exercise 9", r)

### Exercise 10 · Moving average

In [ ]:
def moving_average(a, k):
    c = np.cumsum(np.insert(a.astype(float), 0, 0.0))
    return (c[k:] - c[:-k]) / k

In [ ]:
assert np.allclose(moving_average(np.array([1, 2, 3, 4, 5.]), 3), [2, 3, 4])
assert len(moving_average(np.arange(100.), 7)) == 94
print("✓ exercise 10")